# Connections Analysis — simple workflow

The **SQLite database** (`data/raw/connections.db`) is the source of truth. Use `python project.py scrape` to update it from the web; `python project.py build --semantic` to rebuild all analysis tables. The notebook reads the generated tables.

Only **PyTorch** trains predictive neural models. Chronological **80% train / 20% validation**, with **no test split**. sklearn is used for clustering and descriptive analysis only.


## 1. Load data

Run `python project.py scrape --start YYYY-MM-DD --end YYYY-MM-DD` in a terminal if you need to collect new dates, then `python project.py build --semantic`. The scraper depends on a third-party website and may need updating if its layout changes.

In [1]:
from IPython.display import display
from connections.data import load_data, puzzle_examples, quality_report, puzzle_features
from connections.clustering import load_encoder
words, groups, pairs = load_data()
print(f"Puzzles: {words.puzzle_date.nunique():,}")
display(quality_report(words).head())

Puzzles: 1,219


,puzzle_date,word_rows,unique_words,groups,difficulty,valid_shape
0,2023-06-12,16,16,4,NaN,True
1,2023-06-13,16,16,4,NaN,True
2,2023-06-14,16,16,4,NaN,True
3,2023-06-15,16,16,4,NaN,True
4,2023-06-16,16,16,4,NaN,True


## 2. Descriptive analysis

In [2]:
from connections.analysis import difficulty_trends, color_summary, difficulty_correlations, puzzle_structure, vocabulary_tables
puzzles = puzzle_features(words, pairs)
display(difficulty_trends(puzzles))
display(color_summary(groups))
display(difficulty_correlations(puzzles))
coordinates, clusters, variance = puzzle_structure(puzzles)
display(clusters)
word_counts, category_counts = vocabulary_tables(words)
display(word_counts.sort_values("appearances", ascending=False).head(10))

,year,count,mean,median
0,2024,223,2.939462,3.0
1,2025,365,2.742740,2.8
2,2026,285,2.262105,2.3


,color,count,mean,std
0,blue,1219,0.293641,0.068861
1,green,1219,0.316568,0.074187
2,purple,1219,0.263698,0.059121
3,yellow,1219,0.331251,0.081190


,feature,n,spearman_correlation,p_value
0,within_group_similarity,873,0.013368,0.693255
1,cross_group_similarity,873,0.129014,0.000132
2,similarity_gap,873,-0.097495,0.003934
3,mean_word_length,873,-0.122662,0.000280
4,sd_word_length,873,-0.094480,0.005209


,cluster,puzzles,mean_difficulty,mean_within_similarity,mean_cross_similarity
0,0,227,2.564186,0.260916,0.210915
1,1,991,2.659574,0.309833,0.254104
2,2,1,NaN,1.000000,1.000000


,word,appearances,puzzles_used
464,BALL,30,30
6421,RING,25,25
2927,FLY,22,22
504,BAR,22,22
1348,CHECK,21,21
4340,LEAD,20,20
759,BLUE,19,19
4009,JACK,19,19
8642,WING,19,19
7409,STICK,18,18


## 3. PyTorch grouping solver: synonyms on vs off

Two independent training runs; both report **training and validation** pair AUC and exact puzzle accuracy. First run downloads the embedding model. WordNet is optional: `python -m nltk.downloader wordnet omw-1.4`.

In [3]:
from connections.ml import fit_pair_mlp, solve_with_mlp
import pandas as pd
examples = puzzle_examples(words)
encoder = load_encoder()
models = {}
metrics = []
for use_synonyms in (True, False):
    name = "with_synonyms" if use_synonyms else "without_synonyms"
    print(f"Training {name}")
    model, scores, validation_groups = fit_pair_mlp(examples, encoder, epochs=30, use_synonyms=use_synonyms)
    models[name] = model
    metrics.append(scores.assign(variant=name))
    print("Training performed")
comparison = pd.concat(metrics, ignore_index=True)
display(comparison.round(3))

/home/lukas/Documents/python/miniconda3/envs/BERN_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 69/69 [00:48<00:00,  1.42it/s]


,model,train_pair_auc,validation_pair_auc,train_exact_puzzle_accuracy,validation_exact_puzzle_accuracy,training_puzzles,validation_puzzles,variant
0,PyTorch contextual attention + pooling,0.794,0.681,0.087,0.008,974,244,with_synonyms
1,PyTorch contextual attention + pooling,0.804,0.682,0.102,0.008,974,244,without_synonyms


## 4. PyTorch difficulty regression

Compare the neural regressor with a non-ML training-mean reference; only puzzles with published difficulty ratings contribute to the MAE.

In [4]:
from connections.ml import fit_difficulty_mlp, predict_difficulty
difficulty_model, difficulty_metrics = fit_difficulty_mlp(examples, encoder)
display(difficulty_metrics.round(3))

Batches: 100%|██████████| 69/69 [00:41<00:00,  1.65it/s]


,model,validation_MAE,train_rated,validation_rated
0,PyTorch difficulty MLP,0.819,629,244
1,Training-mean baseline,0.643,629,244


## 5. Unsupervised balanced K-means baseline

This is a clustering baseline (not a sklearn-trained predictive classifier).

In [5]:
from connections.clustering import solve_puzzle
example_words = examples[-1]["words"]
print("PyTorch:", solve_with_mlp(example_words, models["with_synonyms"], encoder))
print("K-means:", solve_puzzle(example_words, encoder))

PyTorch: [['ABBREVIATION', 'CONTRACTION', 'ELISION', 'CONDITION'], ['NICKNAME', 'HANDLE', 'STAND', 'SHAPE'], ['BEAR', 'WEATHER', 'SITUATION', 'STATE'], ['ALKYLAMIDE', 'CALAMARI', 'MANDARIN', 'MELAMINE']]
K-means: [['BEAR', 'HANDLE', 'STAND', 'SHAPE'], ['ALKYLAMIDE', 'CALAMARI', 'MANDARIN', 'MELAMINE'], ['ABBREVIATION', 'CONTRACTION', 'ELISION', 'NICKNAME'], ['WEATHER', 'CONDITION', 'SITUATION', 'STATE']]


## Interpretation

Validation data is used for model selection and early stopping, so validation scores are not independent test-set estimates. No performance improvement is assumed without executing the notebook. Scraping requires internet access and source pages may change.

In [8]:
new_words = [
    "DOG", "CAT", "HORSE", "COW",
    "RED", "BLUE", "GREEN", "YELLOW",
    "APPLE", "BANANA", "ORANGE", "PEAR",
    "CAR", "BUS", "TRAIN", "PLANE"
]
new_words = examples[-22]["words"]
for model in models:
    groups = solve_with_mlp(new_words, models[model], encoder)

    for i, group in enumerate(groups, 1):
        print(f"Group {i}: {group}")

Group 1: ['CAPTURE', 'CHECK', 'SPINNER', 'CHOP']
Group 2: ['CASTLE', 'COMPOSE', 'DRAFTS', 'SENT']
Group 3: ['MATE', 'CHUM', 'BRAH', 'VIVA']
Group 4: ['FLY', 'WORM', 'BEET', 'SPAM']
Group 1: ['CAPTURE', 'FLY', 'SPINNER', 'CHOP']
Group 2: ['CASTLE', 'WORM', 'BEET', 'SPAM']
Group 3: ['CHECK', 'COMPOSE', 'DRAFTS', 'SENT']
Group 4: ['MATE', 'CHUM', 'BRAH', 'VIVA']
